# Chapitre 7 — Comprendre le calcul d'un réseau de neurones · Notebook de travail

*Maths pour l'IA, pas à pas*, Belk Siri — notebook compagnon.

[Ouvrir dans Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/fr/ch07_a_completer.ipynb) · Page compagnon : https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/

Les cellules à modifier portent la mention **À COMPLÉTER**. Exécutez les cellules dans l'ordre : tant qu'une réponse manque, les vérifications le signalent sans erreur. Le corrigé se trouve dans `ch07_corrige.ipynb`, sur la même page compagnon.

**Sommaire.** Données · Les calculs des leçons 7.1 à 7.3 · Leçon 7.4 (mini-lots et époques) · Atelier facultatif (douze messages) · Exercices 7.1 à 7.6

**Avant d'ouvrir Jupyter :** faites d'abord à la main le trajet aller de M7 (leçon 7.1) et la dérivée par rapport à $w_{11}$ (leçon 7.2). Les cellules ci-dessous servent à **vérifier** ces calculs.

**Repères :** indigo = calcul vers l'avant ; magenta en tirets = trajet du gradient ; trait épais ambre = poids étudié $w_{11}$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Repères du livre : calcul vers l'avant en indigo, gradient en magenta (tirets), poids étudié en ambre épais
FORWARD, BACKWARD, STUDIED_WEIGHT = '#3949ab', '#c2185b', '#f9a825'


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def loss(y, p):
    # perte logarithmique d'un message (équation 5.6)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)))


def forward(P, x):
    # Trajet aller (équations 7.1 et 7.2) : renvoie un dictionnaire z1, a1, z2, a2, z, p
    z1 = P['w11'] * x[0] + P['w12'] * x[1] + P['b1']
    z2 = P['w21'] * x[0] + P['w22'] * x[1] + P['b2']
    a1, a2 = sigmoid(z1), sigmoid(z2)
    z = P['v1'] * a1 + P['v2'] * a2 + P['c']
    return {'z1': z1, 'a1': a1, 'z2': z2, 'a2': a2, 'z': z, 'p': sigmoid(z)}


def draw(P, x, backward=False):
    # Le schéma du livre, en plus simple : indigo pour l'aller, magenta pour le retour, w11 en ambre épais
    v = forward(P, x)
    pos = {'x1': (0, 3), 'x2': (0, 0), 'n1': (3, 3), 'n2': (3, 0), 's': (6, 1.5)}
    fig, ax = plt.subplots(figsize=(7, 3.2))
    for a, b, name in (('x1', 'n1', 'w11'), ('x2', 'n1', 'w12'), ('x1', 'n2', 'w21'), ('x2', 'n2', 'w22'),
                      ('n1', 's', 'v1'), ('n2', 's', 'v2')):
        (xa, ya), (xb, yb) = pos[a], pos[b]
        if name == 'w11':
            ax.plot([xa, xb], [ya, yb], color=STUDIED_WEIGHT, linewidth=7, zorder=1)
        ax.annotate('', xy=(xb - 0.35, yb + (ya - yb) * 0.12), xytext=(xa + 0.3, ya),
                    arrowprops=dict(arrowstyle='-|>', color=FORWARD), zorder=2)
        t = 0.3 if a[0] == 'x' and b != ('n1' if a == 'x1' else 'n2') else 0.5
        ax.text(xa + t * (xb - xa), ya + t * (yb - ya) + 0.15, f"{name} = {P[name]:g}" + (' (poids étudié)' if name == 'w11' else ''),
                color=FORWARD, fontsize=8, ha='center', bbox=dict(facecolor='white', edgecolor='none', alpha=0.85))
    texts = {'x1': f"x1 = {x[0]}", 'x2': f"x2 = {x[1]}",
              'n1': f"N1\nz1 = {v['z1']:.3f}\na1 = {v['a1']:.4f}", 'n2': f"N2\nz2 = {v['z2']:.3f}\na2 = {v['a2']:.4f}",
              's': f"sortie\nz = {v['z']:.3f}\np̂ = {v['p']:.3f}"}
    for k, (cx, cy) in pos.items():
        ax.text(cx, cy, texts[k], ha='center', va='center', fontsize=8,
                bbox=dict(boxstyle='circle', facecolor='white', edgecolor=FORWARD if k[0] != 'x' else 'grey'))
    if backward:
        for (x0, y0), (x1, y1), lab in (((6.6, 2.4), (3.5, 3.6), '② v1'), ((3.0, 3.9), (1.4, 3.4), '③ a1(1 − a1), ④ x1')):
            ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                        arrowprops=dict(arrowstyle='-|>', color=BACKWARD, linestyle='--', connectionstyle='arc3,rad=0.3'))
            ax.text((x0 + x1) / 2, max(y0, y1) + 0.35, lab, color=BACKWARD, fontsize=8, ha='center')
        ax.text(6.6, 2.7, '① p̂ − y', color=BACKWARD, fontsize=8)
    ax.set_xlim(-0.8, 7.4); ax.set_ylim(-0.8, 4.4); ax.axis('off'); plt.show()

## Données

Les identifiants D7.1, D7.E1 et D5.1 sont ceux du livre et des fichiers CSV.

In [ ]:
# Données du chapitre 7, recopiées des fichiers CSV du livre (ne pas modifier)

# D7.1 — les neuf paramètres du réseau des leçons 7.1 à 7.3
network = {'w11': 0.5, 'w12': 0.5, 'b1': -2, 'w21': 1, 'w22': -1, 'b2': 0, 'v1': 2, 'v2': -2, 'c': 0}

# D7.E1 — le réseau du collègue (exercices 7.1 et 7.2)
colleague = {'w11': 1, 'w12': 1, 'b1': -4, 'w21': -1, 'w22': 2, 'b2': -1, 'v1': 2, 'v2': 1, 'c': -2}

# D5.1 — les douze messages du chapitre 5 : étiquette, liens, mots suspects
twelve = pd.DataFrame([
    ('M1', 'spam', 5, 4),
    ('M2', 'legitime', 2, 0),
    ('M3', 'legitime', 4, 0),
    ('M4', 'spam', 3, 3),
    ('M5', 'legitime', 1, 0),
    ('M6', 'legitime', 2, 2),
    ('M7', 'spam', 2, 3),
    ('M8', 'legitime', 2, 1),
    ('M9', 'spam', 4, 4),
    ('M10', 'legitime', 1, 1),
    ('M11', 'legitime', 0, 0),
    ('M12', 'legitime', 0, 1)
], columns=['message', 'label', 'links', 'suspicious_words'])

M7 = (2, 3)   # le message M7 : 2 liens, 3 mots suspects, spam (y = 1)
M3 = (4, 0)   # la lettre d'information M3 : 4 liens, 0 mot suspect, légitime (y = 0)

### Les variables des données

Les noms de code sont en anglais. Pour les colonnes, la dernière colonne donne le nom utilisé dans les fichiers CSV du livre.

| Nom | Sens | Unité | Colonne du CSV |
|:--|:--|:--|:--|
| `message` | identifiant du message (M1…) | — | — |
| `label` | étiquette réelle : spam ou légitime | — | `etiquette` |
| `links` | nombre de liens dans le message | liens | `liens` |
| `suspicious_words` | nombre de mots suspects dans le message | mots | `mots_suspects` |
| `network` | les neuf paramètres du réseau de la leçon 7.1 (poids, biais, constante) | — | — |
| `colleague` | les neuf paramètres du réseau du collègue | — | — |
| `twelve` | les douze messages du chapitre 5 | — | — |

## Les calculs des leçons, en Python

Ces cellules refont les calculs des leçons 7.1 à 7.3.

In [ ]:
# Leçon 7.1 — trajet aller du message M7 (équations 7.1 et 7.2)
v = forward(network, M7)
print(f"N1 : z1 = {v['z1']:.3f}, a1 = {v['a1']:.4f}")
print(f"N2 : z2 = {v['z2']:.3f}, a2 = {v['a2']:.4f}")
print(f"sortie : z = {v['z']:.3f}, p̂ = {v['p']:.3f}, perte = {loss(1, v['p']):.3f}")
print(f"M3 (lettre d'information) : a2 = {forward(network, M3)['a2']:.3f}, p̂ = {forward(network, M3)['p']:.3f}")
draw(network, M7)

In [ ]:
# Leçon 7.2 — la règle de la chaîne pour w11 (équation 7.3), facteur par facteur
f1 = v['p'] - 1                  # ① perte ← z : p̂ − y
f2 = network['v1']                # ② z ← a1 : v1
f3 = v['a1'] * (1 - v['a1'])     # ③ a1 ← z1 : dérivée de la sigmoïde, avec l'ACTIVATION a1
f4 = M7[0]                       # ④ z1 ← w11 : x1
d_w11 = f1 * f2 * f3 * f4
print(f"① {f1:.3f}  ② {f2:g}  ③ {f3:.4f}  ④ {f4}  → dérivée = {d_w11:.3f}")

# Vérification par une petite perturbation numérique (comme les sécantes de la leçon 3.3)
def loss_w11(w):
    return loss(1, forward(dict(network, w11=w), M7)['p'])
h = 0.0001
print(f"dérivée numérique : ({loss_w11(0.5 + h):.6f} − {loss_w11(0.5 - h):.6f}) / (2h) = "
      f"{(loss_w11(0.5 + h) - loss_w11(0.5 - h)) / (2 * h):.4f}")

# Une mise à jour à la main, η = 0,5, puis on refait le trajet aller
eta = 0.5
w11_new = round(0.5 - eta * d_w11, 3)   # arrondi au millième, comme dans le livre : 0,655
v_n = forward(dict(network, w11=w11_new), M7)
print(f"w11 : 0,5 → {w11_new:.3f} ; z1 = {v_n['z1']:.2f}, a1 = {v_n['a1']:.4f}, z = {v_n['z']:.3f}, "
      f"p̂ = {v_n['p']:.3f}, perte = {loss(1, v_n['p']):.3f}")
draw(network, M7, backward=True)

ws = np.linspace(-1, 3, 200)
plt.figure(figsize=(6, 3))
plt.plot(ws, [loss_w11(w) for w in ws], color='dimgrey', label='perte de M7 selon w11')
plt.plot(ws[25:126], loss_w11(0.5) + d_w11 * (ws[25:126] - 0.5), '--', color=BACKWARD, label='tangente en 0,5')
plt.scatter([0.5, w11_new], [loss_w11(0.5), loss_w11(w11_new)], color=['black', FORWARD], zorder=3)
plt.xlabel('w11'); plt.ylabel('perte de M7'); plt.legend(); plt.grid(alpha=0.4); plt.show()

In [ ]:
# Leçon 7.3 — rétropropagation : les neuf dérivées (équation 7.4), puis une mise à jour de TOUS les paramètres
def backward(P, x, y):
    v = forward(P, x)
    d = v['p'] - y                                   # signal d'erreur de la sortie
    d1 = d * P['v1'] * v['a1'] * (1 - v['a1'])       # signal d'erreur de N1
    d2 = d * P['v2'] * v['a2'] * (1 - v['a2'])       # signal d'erreur de N2
    return {'w11': d1 * x[0], 'w12': d1 * x[1], 'b1': d1, 'w21': d2 * x[0], 'w22': d2 * x[1], 'b2': d2,
            'v1': d * v['a1'], 'v2': d * v['a2'], 'c': d}


def update(P, g, eta):
    return {k: P[k] - eta * g[k] for k in P}        # toutes les dérivées sont prises AVANT de modifier


g = backward(network, M7, 1)
new = update(network, g, 0.5)
print(pd.DataFrame({'valeur': network, 'dérivée': g, 'nouvelle valeur': new}).round(3))
print(f"M7 : p̂ = {forward(new, M7)['p']:.3f}, perte = {loss(1, forward(new, M7)['p']):.3f}")
print(f"M3 avant : p̂ = {forward(network, M3)['p']:.3f}, perte = {loss(0, forward(network, M3)['p']):.3f} ; "
      f"après : p̂ = {forward(new, M3)['p']:.3f}, perte = {loss(0, forward(new, M3)['p']):.3f}")

## Leçon 7.4 — Apprendre par petits lots : mini-lots et époques

Réseau dans son état de départ (D7.1), douze messages du chapitre 5. Pour suivre le calcul à la main, seule la constante de sortie $c$ bouge. Dérivée d'un lot = moyenne des dérivées de ses messages (équation 7.5).

In [ ]:
# Leçon 7.4 — somme s, probabilité et signal d'erreur δ = p̂ − y des douze messages, au départ (c = 0)
X12 = list(zip(twelve['links'], twelve['suspicious_words']))
y12 = [1 if e == 'spam' else 0 for e in twelve['label']]


def p_with_c(x, c_out):
    return forward(dict(network, c=c_out), x)['p']    # seule la constante de sortie change


deltas = np.array([p_with_c(x, 0) - y_ for x, y_ in zip(X12, y12)])
print(pd.DataFrame({'message': twelve['message'], 'y': y12, 's': [forward(network, x)['z'] for x in X12],
                    'p̂': [p_with_c(x, 0) for x in X12], 'δ': deltas}).round(3).to_string(index=False))
print(f"dérivée par rapport à c — M7 seul : {deltas[6]:.3f} ; lot M1 à M4 : {deltas[:4].mean():.3f} ; "
      f"douze : somme {deltas.sum():.3f}, moyenne {deltas.mean():.3f}")
print(f"lots M5 à M8 : {deltas[4:8].mean():.3f} ; M9 à M12 : {deltas[8:].mean():.3f} ; "
      f"moyenne des trois lots : {np.mean([deltas[:4].mean(), deltas[4:8].mean(), deltas[8:].mean()]):.3f}")


def loss_c(c_out):
    return np.mean([loss(y_, p_with_c(x, c_out)) for x, y_ in zip(X12, y12)])


# Une époque : trois mises à jour, chacune sur son lot, chacune partant de la valeur laissée par la précédente
c_batch, path = 0.0, [0.0]
for k in range(3):
    batch = range(4 * k, 4 * k + 4)
    d = np.mean([p_with_c(X12[i], c_batch) - y12[i] for i in batch])
    c_batch = c_batch - 0.5 * d
    path.append(c_batch)
    print(f"mise à jour {k + 1} (M{4 * k + 1} à M{4 * k + 4}) : dérivée {d:.4f} → c = {c_batch:.3f}")
c_one_step = 0 - 0.5 * deltas.mean()
print(f"perte moyenne des douze : {loss_c(0):.3f} → {loss_c(c_batch):.3f} après une époque (3 mises à jour) ; "
      f"un seul pas sur les douze : c = {c_one_step:.3f}, perte {loss_c(c_one_step):.3f}")
plt.figure(figsize=(6, 2.6))
plt.plot(range(4), path, marker='o', color=FORWARD, label='une époque, lots de quatre')
plt.plot([0, 1], [0, c_one_step], marker='s', color='black', label='un pas sur les douze')
plt.plot([0, 1], [0, -0.5 * deltas[6]], marker='^', color=BACKWARD, linestyle='--', label='M7 seul')
plt.xlabel('mise à jour'); plt.ylabel('constante c'); plt.legend(); plt.grid(alpha=0.4); plt.show()

In [ ]:
# Questions courtes (leçon 7.4) — À COMPLÉTER
updates_per_epoch = None   # À COMPLÉTER : mises à jour par époque, douze messages en lots de TROIS
updates_ten_epochs = None  # À COMPLÉTER : mises à jour pour dix époques
c_batch_M6_M7 = None      # À COMPLÉTER : nouvelle valeur de c après une mise à jour sur le lot {M6, M7}, η = 0,5

In [ ]:
# Vérification des questions courtes
if updates_per_epoch is None or updates_ten_epochs is None or c_batch_M6_M7 is None:
    print('pas encore complété. Complétez les trois lignes.')
else:
    print(f'1. {updates_per_epoch} par époque, {updates_ten_epochs} pour dix époques →',
          'correct' if (updates_per_epoch, updates_ten_epochs) == (4, 40) else 'à revoir')
    print(f'2. c = {c_batch_M6_M7:.3f} →', 'correct' if abs(c_batch_M6_M7 + 0.5 * deltas[5:7].mean()) < 1e-9 else 'à revoir')

## Atelier facultatif — entraîner le réseau sur les douze messages

1 000 pas de descente de gradient ($\eta = 0{,}5$), à partir des poids de la leçon 7.1, en moyennant à chaque pas les dérivées des douze messages du chapitre 5. **Toutes les mesures ci-dessous portent sur ces douze messages, qui servent à l'entraînement** : elles ne disent rien du comportement du réseau sur des messages nouveaux (leçon 6.3).

In [ ]:
# Atelier facultatif : entraînement sur les douze messages (mesures sur les données d'entraînement)
X12 = list(zip(twelve['links'], twelve['suspicious_words']))
y12 = [1 if e == 'spam' else 0 for e in twelve['label']]


def loss_12(P):
    return np.mean([loss(y, forward(P, x)['p']) for x, y in zip(X12, y12)])


def gradient_12(P):
    gs = [backward(P, x, y) for x, y in zip(X12, y12)]
    return {k: np.mean([g[k] for g in gs]) for k in P}     # dérivée de la perte moyenne = moyenne des dérivées


P_trained, history = dict(network), [loss_12(network)]
for _ in range(1000):
    P_trained = update(P_trained, gradient_12(P_trained), 0.5)
    history.append(loss_12(P_trained))
print(f"perte moyenne sur les douze messages d'entraînement : avant {history[0]:.3f}, après {history[-1]:.3f}")
for name, x in (('M7', M7), ('M3', M3)):
    print(f"{name} : p̂ avant {forward(network, x)['p']:.3f}, après {forward(P_trained, x)['p']:.3f}")
print('erreurs au seuil 0,5 sur les douze : avant', sum((forward(network, x)['p'] >= 0.5) != y for x, y in zip(X12, y12)),
      ', après', sum((forward(P_trained, x)['p'] >= 0.5) != y for x, y in zip(X12, y12)))
print('poids après entraînement :', {k: round(float(v), 2) for k, v in P_trained.items()})
plt.figure(figsize=(6, 3)); plt.plot(history, color=FORWARD)
plt.xlabel('pas'); plt.ylabel("perte moyenne (douze messages d'entraînement)"); plt.grid(alpha=0.4); plt.show()

## Exercice 7.1 — Lire le réseau d'un collègue

*Notions : couches, poids et biais (leçon 7.1, équations 7.1 et 7.2).* Données D7.E1 : le dictionnaire `colleague`.

### a) à e) Sur papier, en lisant la figure du livre

*Vos réponses :* …

In [ ]:
# À COMPLÉTER : nombre de paramètres du réseau du collègue, et sommes pondérées d'un message vide
n_parameters = None   # À COMPLÉTER : len(...)
empty = None            # À COMPLÉTER : forward(colleague, (0, 0))

In [ ]:
# Vérification de a) et d)
if n_parameters is None or empty is None:
    print('pas encore complété. Complétez les deux lignes.')
else:
    print(f"{n_parameters} paramètres ; message vide : z1 = {empty['z1']:g}, z2 = {empty['z2']:g} →",
          'correct' if n_parameters == 9 and empty['z1'] == -4 and empty['z2'] == -1 else 'à revoir')

## Exercice 7.2 — Calculer une sortie à la main

*Notions : trajet aller (équations 7.1 et 7.2), sigmoïde (équation 5.4), perte (équation 5.6), règle de décision (équation 5.1).*

### a) à c) Sur papier

*Vos calculs :* …

In [ ]:
# À COMPLÉTER : probabilité et perte de M6 (légitime, y = 0) et de M7 (spam, y = 1) dans le réseau du collègue
p_M6 = None         # À COMPLÉTER : utilisez forward(colleague, (2, 2))['p']
loss_M6 = None     # À COMPLÉTER : utilisez loss(y, p)
loss_M7 = None     # À COMPLÉTER

In [ ]:
# Vérification de a) à c)
if p_M6 is None or loss_M6 is None or loss_M7 is None:
    print('pas encore complété. Complétez les trois lignes.')
else:
    print(f"M6 : p̂ = {p_M6:.3f}, signalé : {p_M6 >= 0.5}, perte = {loss_M6:.3f} →",
          'correct' if abs(p_M6 - 0.4332) < 5e-4 and abs(loss_M6 - 0.5677) < 5e-4 else 'à revoir')
    print(f"M7 : perte {loss_M7:.3f} (collègue) contre {loss(1, forward(network, M7)['p']):.3f} (leçon 7.1) →",
          'correct' if abs(loss_M7 - 0.5071) < 5e-4 else 'à revoir')
    draw(colleague, (2, 2))

## Exercice 7.3 — Interpréter une activation

*Notions : rôle d'un neurone caché (leçon 7.1).* Réseau de la leçon 7.1, douze messages du chapitre 5.

### a) à d) Sur papier

*Vos réponses :* …

In [ ]:
# À COMPLÉTER : activation a2 de chaque message, puis la liste des messages où elle dépasse 0,8
twelve['a2'] = None    # À COMPLÉTER : [forward(network, (l, m))['a2'] for l, m in zip(twelve['links'], twelve['suspicious_words'])]
lit_up = None        # À COMPLÉTER

In [ ]:
# Vérification de a) et c)
if lit_up is None or twelve['a2'].isna().any():
    print('pas encore complété. Complétez les deux lignes.')
else:
    twelve['a1'] = [forward(network, (l, m))['a1'] for l, m in zip(twelve['links'], twelve['suspicious_words'])]
    twelve['p̂'] = [forward(network, (l, m))['p'] for l, m in zip(twelve['links'], twelve['suspicious_words'])]
    print(twelve.round(3).to_string(index=False))
    print('a2 > 0,8 :', lit_up, '→', 'correct' if lit_up == ['M2', 'M3'] else 'à revoir')
    print(f"M6 : z = {forward(network, (2, 2))['z']}, p̂ = {forward(network, (2, 2))['p']} → signalé au seuil 0,5 : {forward(network, (2, 2))['p'] >= 0.5}")

## Exercice 7.4 — Compléter une dérivée dans Jupyter

*Notions : règle de la chaîne (équation 7.3), vérification numérique (leçon 7.2).* Le poids $w_{22}$ relie les mots suspects au neurone N2.

### a) et b) Sur papier

*Vos quatre facteurs et votre explication :* …

In [ ]:
# À COMPLÉTER : dérivée de la perte par rapport à w22, par la règle de la chaîne
def derivative_w22(x, y):
    v = forward(network, x)
    return None   # À COMPLÉTER : produit des quatre facteurs (p̂ − y) × v2 × a2(1 − a2) × x2
derivee_w22 = derivative_w22   # alias du livre (nom de l'édition française imprimée)

In [ ]:
# Vérification de c) : règle de la chaîne contre perturbation numérique
if derivative_w22(M7, 1) is None:
    print('pas encore complété. Complétez la fonction.')
else:
    h = 0.0001
    num = (loss(1, forward(dict(network, w22=-1 + h), M7)['p']) - loss(1, forward(dict(network, w22=-1 - h), M7)['p'])) / (2 * h)
    print(f"règle de la chaîne : {derivative_w22(M7, 1):.4f} →", 'correct' if abs(derivative_w22(M7, 1) - 0.3896) < 5e-4 else 'à revoir')
    print(f"dérivée numérique  : {num:.4f} → écart {abs(num - derivative_w22(M7, 1)):.1e} →",
          'correct' if abs(num - derivative_w22(M7, 1)) < 1e-6 else 'à revoir')

## Exercice 7.5 — Prédire, puis vérifier : l'effet d'un changement de poids

*Notions : dérivée et approximation (leçons 3.3 et 7.2).*

### a) Votre prédiction

*Pertes prédites pour $w_{11} = 0{,}6$ et $w_{11} = 2{,}5$ :* …

In [ ]:
# À COMPLÉTER : perte exacte de M7 après un changement dw du seul poids w11
def loss_after(dw):
    return None   # À COMPLÉTER : refaites le trajet aller avec dict(network, w11=0.5 + dw)
perte_apres = loss_after   # alias du livre (nom de l'édition française imprimée)

In [ ]:
# Vérification de b)
if loss_after(0.1) is None:
    print('pas encore complété. Complétez la fonction.')
else:
    L0 = loss(1, forward(network, M7)['p'])
    for dw in (0.1, 2):
        print(f"dw = {dw} : prédite {L0 + d_w11 * dw:.3f}, exacte {loss_after(dw):.3f}")
    print('→', 'correct' if abs(loss_after(0.1) - 0.3716) < 5e-4 and abs(loss_after(2) - 0.2126) < 5e-4 else 'à revoir')

### c) Votre interprétation

…

## Exercice 7.6 — Défi : mettre à jour plusieurs poids sur deux messages

*Notions : rétropropagation (équation 7.4), perte moyenne (équation 5.7), descente de gradient (équation 3.7).* Messages M7 (spam) et M3 (légitime).

### a) à c) Sur papier

*Vos calculs :* …

In [ ]:
# À COMPLÉTER : dérivées de la PERTE MOYENNE des deux messages (moyenne des dérivées de chaque message)
def mean_gradient(P):
    g7, g3 = backward(P, M7, 1), backward(P, M3, 0)
    return None   # À COMPLÉTER : un dictionnaire {paramètre: moyenne des deux dérivées}
gradient_moyen = mean_gradient   # alias du livre (nom de l'édition française imprimée)

In [ ]:
# Vérification de c), puis d) : les neuf paramètres, un pas puis 100 pas
def mean_loss(P):
    return (loss(1, forward(P, M7)['p']) + loss(0, forward(P, M3)['p'])) / 2

if mean_gradient(network) is None:
    print('pas encore complété. Complétez la fonction.')
else:
    g = mean_gradient(network)
    output_only = dict(network, v1=network['v1'] - g['v1'], v2=network['v2'] - g['v2'], c=network['c'] - g['c'])
    print(f"départ : perte moyenne {mean_loss(network):.3f}")
    print(f"c) v1, v2, c seulement : M7 {loss(1, forward(output_only, M7)['p']):.3f}, "
          f"M3 {loss(0, forward(output_only, M3)['p']):.3f}, moyenne {mean_loss(output_only):.3f} →",
          'correct' if abs(mean_loss(output_only) - 0.3519) < 5e-4 else 'à revoir')
    nine = update(network, g, 1)
    print(f"d) les neuf paramètres, un pas : moyenne {mean_loss(nine):.3f} →",
          'correct' if abs(mean_loss(nine) - 0.2584) < 5e-4 else 'à revoir')
    P, history = dict(network), []
    for _ in range(100):
        P = update(P, mean_gradient(P), 1)
        history.append(mean_loss(P))
    print(f"   100 pas : p̂(M7) = {forward(P, M7)['p']:.3f}, p̂(M3) = {forward(P, M3)['p']:.3f}, perte moyenne {history[-1]:.3f}")
    plt.figure(figsize=(6, 3)); plt.plot(range(1, 101), history, color=FORWARD)
    plt.xlabel('pas'); plt.ylabel('perte moyenne (M7 et M3)'); plt.grid(alpha=0.4); plt.show()

### d) Votre conclusion

…